# Module 2.1: Deploying the Self-Hosted UniFi Network Application on `homelab-gpu`

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 2: Managed Switch Control Plane

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand Ubiquiti's **Software-Defined Networking (SDN)** architecture—specifically why the `USW-Enterprise-8-PoE` has no standalone local web UI and instead pairs with a **UniFi Network Application** controller.
2. Explain the hardware engineering reason we host the UniFi Network Application + **MongoDB 7.0** on **`homelab-gpu` (`x86_64` with AVX/AVX2, 16–32 GB RAM)** rather than on the 4 GB ARM Raspberry Pi 4.
3. Deploy the modern two-container stack (`lscr.io/linuxserver/unifi-network-application` + pinned `docker.io/mongo:7.0` with `init-mongo.sh`) on `homelab-gpu` (`10.0.0.201:8443`).
4. Complete the local-only administrator setup wizard and configure the controller **Inform Host Override** (`10.0.0.201`) so switches can cleanly adopt over the LAN.

## 2.1.1 Conceptual Background: Unlocking Your Enterprise Switch Without Buying a Cloud Key

Unlike consumer smart switches (such as a TP-Link TL-SG108E) that embed a tiny local HTTP page on the switch itself, Ubiquiti UniFi switches operate under a **centralized SDN control-plane model**:
- The switch (`USW-Enterprise-8-PoE`) handles **data-plane forwarding** in dedicated ASIC hardware at 80 Gbps wire speed—even if the controller is offline or rebooting!
- The **UniFi Network Application** runs the **control plane**: pushing port labels, PoE on/off toggles, firmware updates, RSTP loop protection, VLAN profiles, and collecting live per-port link speed, packet error, wattage, and temperature telemetry.
- You do **not** need to buy a $200+ UniFi Cloud Key or UniFi Gateway—Ubiquiti officially supports self-hosting the UniFi Network Application for free in Docker!

### 1. Why Host UniFi Network Application on `homelab-gpu` Instead of `homelab-pi`?

| Engineering Factor | `homelab-pi` (Raspberry Pi 4, 4 GB ARM64) | `homelab-gpu` (2019 ROG Zephyrus, `x86_64`) | Winner |
| :--- | :--- | :--- | :---: |
| **MongoDB 7.0 / 8.0 CPU ISA Requirement** | Cortex-A72 is ARMv8.0-A (lacks ARMv8.2-A instructions required by MongoDB 5.0+, forcing EOL MongoDB 4.4!) | Intel Core i7-9750H / AMD Ryzen 7 3750H natively supports **AVX / AVX2**, running modern **MongoDB 7.0** cleanly | **`homelab-gpu`** |
| **JVM + MongoDB Memory Footprint** | ~700 MB – 1.1 GB RAM (~25% of Pi's 4 GB total RAM, risking OOM alongside Immich Postgres + Pi-hole + Kuma) | ~700 MB – 1.1 GB RAM (<6% of 16–32 GB DDR4 RAM) | **`homelab-gpu`** |
| **Data-Plane Resilience During Reboots** | — | If `homelab-gpu` ever reboots for kernel updates, the `USW-Enterprise-8-PoE` **continues switching 100% of LAN traffic uninterrupted** using its saved flash config! | **Zero Downtime** |

### 2. Required Ports & MongoDB Version-Pinning Rule

```mermaid
flowchart LR
    Browser["Workstation Browser<br>https://10.0.0.201:8443"] -->|TCP 8443 HTTPS UI| UniFi["unifi-network-application<br>(LinuxServer.io Java SDN)"]
    Switch["USW-Enterprise-8-PoE<br>(10.0.0.203)"] <-->|"TCP 8080 (Device Inform)<br>UDP 3478 (STUN)<br>UDP 10001 (L2 Discovery)"| UniFi
    UniFi <-->|"Internal Docker Bridge<br>TCP 27017"| Mongo["unifi-db<br>(Pinned docker.io/mongo:7.0)"]
```

> ⚠️ **CRITICAL MONGODB RULE:** **Never** use `image: mongo:latest` for UniFi! MongoDB does not support skipping major database formats automatically—if an unattended pull jumps from Mongo 7 to Mongo 8 or 9, the database refuses to boot. Always pin an explicit major.minor tag (`docker.io/mongo:7.0`) and initialize dedicated least-privilege database roles using `init-mongo.sh`.

## 2.1.2 Step-by-Step Laboratory Guide

### Task 2.1.1: Create the UniFi Network Stack Directory & `init-mongo.sh` on `homelab-gpu`
1. SSH into your GPU/Compute node (`ssh homelab-gpu`) and create the stack and persistent data directories:
   ```bash
   mkdir -p ~/homelab/stacks/unifi-network/{config,db-data}
   cd ~/homelab/stacks/unifi-network
   ```
2. Verify that your Zephyrus CPU supports `avx` (confirming full MongoDB 7.0 compatibility):
   ```bash
   grep -o -m1 'avx' /proc/cpuinfo
   ```
   *(Expected output: `avx`)*
3. Create the MongoDB initialization script (`init-mongo.sh`) that automatically provisions the `unifi` user and its four required UniFi databases on first boot:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/unifi-network/init-mongo.sh
   #!/bin/bash
   if which mongosh > /dev/null 2>&1; then
     mongo_init_bin='mongosh'
   else
     mongo_init_bin='mongo'
   fi

   "${mongo_init_bin}" <<EOF
   use ${MONGO_AUTHSOURCE}
   db.auth("${MONGO_INITDB_ROOT_USERNAME}", "${MONGO_INITDB_ROOT_PASSWORD}")
   db.createUser({
     user: "${MONGO_USER}",
     pwd: "${MONGO_PASS}",
     roles: [
       "clusterMonitor",
       { db: "${MONGO_DBNAME}", role: "dbOwner" },
       { db: "${MONGO_DBNAME}_stat", role: "dbOwner" },
       { db: "${MONGO_DBNAME}_audit", role: "dbOwner" },
       { db: "${MONGO_DBNAME}_restore", role: "dbOwner" }
     ]
   })
   EOF
   EOT
   chmod +x ~/homelab/stacks/unifi-network/init-mongo.sh
   ```

### Task 2.1.2: Generate Secure Credentials, Write `docker-compose.yml` & Launch UniFi Network Server
1. Generate two strong random passwords and store them in a restricted `.env` file (`chmod 600`):
   ```bash
   cat << EOT > ~/homelab/stacks/unifi-network/.env
   TZ=America/New_York
   PUID=$(id -u)
   PGID=$(id -g)
   MONGO_ROOT_PASS=$(openssl rand -hex 20)
   MONGO_UNIFI_PASS=$(openssl rand -hex 20)
   EOT
   chmod 600 ~/homelab/stacks/unifi-network/.env
   ```
2. Create `~/homelab/stacks/unifi-network/docker-compose.yml`:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/unifi-network/docker-compose.yml
   name: unifi_network

   services:
     unifi-db:
       image: docker.io/mongo:7.0
       container_name: unifi_db
       environment:
         - MONGO_INITDB_ROOT_USERNAME=root
         - MONGO_INITDB_ROOT_PASSWORD=${MONGO_ROOT_PASS}
         - MONGO_USER=unifi
         - MONGO_PASS=${MONGO_UNIFI_PASS}
         - MONGO_DBNAME=unifi
         - MONGO_AUTHSOURCE=admin
       volumes:
         - ./db-data:/data/db
         - ./init-mongo.sh:/docker-entrypoint-initdb.d/init-mongo.sh:ro
       restart: unless-stopped

     unifi-network-application:
       image: lscr.io/linuxserver/unifi-network-application:latest
       container_name: unifi_network_application
       depends_on:
         - unifi-db
       environment:
         - PUID=${PUID}
         - PGID=${PGID}
         - TZ=${TZ}
         - MONGO_USER=unifi
         - MONGO_PASS=${MONGO_UNIFI_PASS}
         - MONGO_HOST=unifi-db
         - MONGO_PORT=27017
         - MONGO_DBNAME=unifi
         - MONGO_AUTHSOURCE=admin
         - MEM_LIMIT=1024
         - MEM_STARTUP=512
       volumes:
         - ./config:/config
       ports:
         - "8443:8443"         # Web Admin UI (HTTPS)
         - "8080:8080"         # Device Inform & Adoption
         - "3478:3478/udp"     # STUN Telemetry
         - "10001:10001/udp"   # Layer-2 Device Discovery
         - "1900:1900/udp"     # L2 Discoverable in Network
         - "6789:6789"         # Mobile Throughput Test (Optional)
       restart: unless-stopped
   EOT
   ```
3. Pull images and start the controller stack:
   ```bash
   cd ~/homelab/stacks/unifi-network
   docker compose pull
   docker compose up -d
   docker compose ps
   ```
4. Wait ~30–45 seconds for the Java application to initialize its MongoDB collections, then verify port `8443` is listening:
   ```bash
   docker logs --tail 25 unifi_network_application
   ```

### Task 2.1.3: Complete First-Time Setup Wizard & Set the Inform Host Override (`10.0.0.201`)
1. Open your web browser and navigate to:
   ```text
   https://10.0.0.201:8443
   ```
   *(Accept the self-signed HTTPS browser warning—this is your private LAN controller.)*
2. Walk through the **UniFi Network Setup Wizard**:
   - **Server Name:** `Homelab UniFi Network`
   - **Region / Country:** Select your country.
   - **Account Setup:** Click **Advanced Setup** $\rightarrow$ uncheck *Remote Access* if you prefer a **100% local administrator account** (recommended for a zero-cloud homelab accessed via Tailscale!), OR sign in with your Ubiquiti UI account if you want `unifi.ui.com` cloud portal integration.
   - Create your local admin username and strong password, then click **Finish**.
3. **CRITICAL DOCKER SETTING — Set the Inform Host IP:**
   Because the UniFi controller runs inside a Docker bridge network (e.g., `172.18.0.x`), adopting a physical switch requires telling the controller to advertise your laptop's real LAN IP (`10.0.0.201`):
   - In the UniFi Network Web UI, navigate to **Settings** (bottom-left gear icon) $\rightarrow$ **System** $\rightarrow$ **Advanced**.
   - Enable **Inform Host** $\rightarrow$ check **Override**, enter **`10.0.0.201`**, and click **Apply Changes**!

### 🖼️ Task 2.1.3 Verification: Self-Hosted UniFi Network Dashboard & Inform Host Override

**Verification Objective:** Capture a screenshot of your UniFi Network Application dashboard (`https://10.0.0.201:8443`) running cleanly on `homelab-gpu` (alongside `docker compose ps` showing both `unifi_db` and `unifi_network_application` Up).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `homelab-gpu` AVX support verified and pinned `mongo:7.0` + `init-mongo.sh` deployed.
- [ ] `unifi_network_application` running cleanly on `https://10.0.0.201:8443` with ports `8080`, `3478/udp`, and `10001/udp` exposed.
- [ ] Initial setup wizard completed with a strong local (or UI) administrator account.
- [ ] **Inform Host Override** set to `10.0.0.201` under **Settings $\rightarrow$ System $\rightarrow$ Advanced**.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 2.2: Adopting `USW-Enterprise-8-PoE`, Port Profiles & PoE Safety Hygiene](./02_2_switch_adoption_port_profiles_and_poe_hygiene.ipynb).